# OCR pre-pass: PP-OCRv5 on every page of a manifest (CPU runtime)

Runs on a **Colab CPU runtime** (Runtime → Change runtime type → CPU; High-RAM gives more cores). OCR runs on the CPU anyway, so doing it here instead of on the L4 keeps the paid GPU busy with model calls only. The text and hybrid paths later read each page's text **and its original OCR time** from the cache, so the cost they report is unchanged; every cache entry records the CPU it was measured on.

`benchmarks/full.csv` has 3,586 distinct pages. The time per page depends on the cores you get: new pages print their time as they finish, and the mean is printed at the end. The cache lives on Google Drive, so the pass is **resumable**: if the session dies, run all cells again and cached pages are skipped.

Meanwhile the GPU runbook (`gpu_run.ipynb`) can already run the **vision** path, which needs no OCR.

In [ ]:
# 1. Settings
MANIFEST = "benchmarks/full.csv"
OCR = "paddle-mobile"  # PP-OCRv5 mobile: answer coverage 0.63 vs Tesseract 0.42 on the pilot pages
DRIVE = "/content/drive/MyDrive/qdu"
OCR_CACHE = f"{DRIVE}/ocr_cache"
RUNS_DIR = f"{DRIVE}/runs"
REPO = "https://github.com/emre-geris/query-based-document-understanding.git"

import os
import subprocess

print(subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip(), "CPU cores")

In [ ]:
# 2. Drive
# Google Drive keeps the OCR cache and the runs across sessions: a Colab runtime loses its disk
# when it disconnects, and the full run spans several sessions.
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(OCR_CACHE, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
# 3. Code, installed with uv (uv brings its own Python 3.13)
token = os.environ.get("GITHUB_TOKEN")
try:
    from google.colab import userdata

    token = token or userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
os.chdir("/content")
if not os.path.isdir("qdu-repo"):
    subprocess.run(["git", "clone", "-q", url, "qdu-repo"], check=True)
os.chdir("/content/qdu-repo")
!git pull -q && git log --oneline -1
!pip -q install -U uv
!uv sync -q --extra paddle && uv run qdu ocr warm --help | head -2

In [ ]:
# 4. Datasets -> HF cache (~1.6 GB, all four)
!uv run qdu data download

In [ ]:
# 5. oneDNN check on 3 pages (scratch cache). oneDNN makes CPU OCR several times faster but
#    crashed on PP-OCRv5 under Windows; if it fails here too, continue without it (the engine
#    name then ends in -nomkldnn, so the slower OCR times stay identifiable).
check = subprocess.run(
    f"uv run qdu ocr warm --engine {OCR} --manifest {MANIFEST} --limit 3 --ocr-cache /content/ocr_check",
    shell=True,
    capture_output=True,
    text=True,
)
if check.returncode == 0:
    print(check.stdout[-600:])
else:
    print(check.stderr[-1500:])
    os.environ["QDU_PADDLE_MKLDNN"] = "0"
    print("oneDNN failed: continuing with QDU_PADDLE_MKLDNN=0; set the same in gpu_run.ipynb")

In [ ]:
# 6. The pre-pass. Resumable: re-run cells 1-5 and this one after a disconnect.
!uv run qdu ocr warm --engine {OCR} --manifest {MANIFEST} --ocr-cache {OCR_CACHE}